In [12]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
import xgboost as xgb
import lightgbm as lgb

In [13]:
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 110

RAW_PATH = r'C:\Users\amare\OneDrive\Desktop\Projects\Finance Hackathon\stdbank-pip\data\01_raw\Synthetic_Financial_Datasets_For_Fraud_Detection_resample.csv'
TOL = 0.01

In [14]:
df = pd.read_csv(r'C:\Users\amare\OneDrive\Desktop\Projects\Finance Hackathon\stdbank-pip\data\01_raw\Synthetic_Financial_Datasets_For_Fraud_Detection_resample.csv')

In [15]:
df

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,182,CASH_IN,100328.70,C197678862,3389602.37,3489931.07,C386191921,219735.14,119406.44,0,0
1,210,PAYMENT,11660.99,C2009511954,94054.00,82393.01,M564284134,0.00,0.00,0,0
2,403,CASH_IN,230751.30,C1132312861,23117012.10,23347763.39,C1025694734,951294.81,720543.51,0,0
3,328,PAYMENT,17833.60,C1191709365,0.00,0.00,M293024801,0.00,0.00,0,0
4,563,CASH_OUT,246476.67,C342438889,0.00,0.00,C2087488957,4226850.16,4473326.82,0,0
...,...,...,...,...,...,...,...,...,...,...,...
636257,325,CASH_OUT,193812.50,C425986045,95438.00,0.00,C1813421380,511348.22,705160.72,0,0
636258,324,PAYMENT,71240.51,C246016542,5021.00,0.00,M332903748,0.00,0.00,0,0
636259,231,CASH_IN,219467.05,C152812322,3990.00,223457.05,C316565040,2593190.49,2373723.44,0,0
636260,306,PAYMENT,1048.54,C37754145,0.00,0.00,M1903551785,0.00,0.00,0,0


In [16]:

TOLERANCE = 0.01

orig_ok = np.isclose(df['newbalanceOrig'],
                     df['oldbalanceOrg'] - df['amount'],
                     atol=TOLERANCE)

dest_ok = np.isclose(df['newbalanceDest'],
                     df['oldbalanceDest'] + df['amount'],
                     atol=TOLERANCE)

# Keep only rows where BOTH sides of the ledger add up
filtered = df[orig_ok & dest_ok].copy()

print(f'Original rows : {len(df):,}')
print(f'Filtered rows : {len(filtered):,}')

Original rows : 636,262
Filtered rows : 27,990


In [17]:
filtered['isFraud'].value_counts()

isFraud
0    27589
1      401
Name: count, dtype: int64

In [18]:
df[df['nameDest']=='C1049817027']

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
89130,14,CASH_IN,246652.81,C347024574,14787.00,261439.81,C1049817027,12648.08,0.00,0,0
231040,131,CASH_OUT,94350.11,C1331987137,10904.00,0.00,C1049817027,443580.22,537930.33,0,0
331481,1,CASH_OUT,32282.57,C123674777,14375.74,0.00,C1049817027,97055.04,60738.03,0,0
358001,43,CASH_OUT,84293.62,C1702265076,104044.27,19750.65,C1049817027,295689.01,379982.63,0,0
479990,1,CASH_IN,25927.54,C1457417579,8151115.24,8177042.78,C1049817027,129337.61,60738.03,0,0


P(oldbalanceOrg == amount | fraud) = 0.9753P(oldbalanceOrg == amount | legit) = 0.0000Mechanism: the fraudster empties the account (amount = full balance), andPaySim cancels fraudulent transactions, so newbalanceOrig stays untouched.A single boolean feature therefore separates the classes almost perfectly.This is why the public leaderboard for this dataset sits at AUC 0.99+.

In [19]:
df[(df["oldbalanceOrg"] == df["amount"]) & (df["isFraud"] == 1)]

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
207,512,CASH_OUT,127120.88,C130072062,127120.88,0.0,C681167784,2605727.43,2732848.31,1,0
256,345,CASH_OUT,128936.95,C1177265377,128936.95,0.0,C1158181997,1741561.22,1870498.17,1,0
922,479,CASH_OUT,799517.33,C1299137401,799517.33,0.0,C466144148,67824.14,867341.47,1,0
1643,719,TRANSFER,119306.06,C1183843461,119306.06,0.0,C1830685403,0.00,0.00,1,0
2153,208,TRANSFER,4815187.09,C2041567241,4815187.09,0.0,C2093303821,0.00,0.00,1,0
...,...,...,...,...,...,...,...,...,...,...,...
634395,395,CASH_OUT,47984.43,C790140463,47984.43,0.0,C2097780995,299049.39,347033.81,1,0
634953,554,TRANSFER,3576297.10,C193696150,3576297.10,3576297.1,C484597480,0.00,0.00,1,1
635532,85,TRANSFER,4094.07,C810305173,4094.07,0.0,C600417404,0.00,0.00,1,0
635657,159,TRANSFER,416832.90,C1395649646,416832.90,0.0,C640393193,0.00,0.00,1,0


# Every fraudulent transaction where the sender empties their account is either a TRANSFER or CASH_OUT

In [20]:
df["type"].unique()

array(['CASH_IN', 'PAYMENT', 'CASH_OUT', 'TRANSFER', 'DEBIT'],
      dtype=object)

In [21]:
# Calculate the expected destination balance after the transfer
df['expected_newbalanceDest'] = df['oldbalanceDest'] + df['amount']

# Calculate the absolute discrepancy error at the destination
df['dest_balance_error'] = df['expected_newbalanceDest'] - df['newbalanceDest']

In [22]:
def engineer_fraud_features(df: pd.DataFrame) -> pd.DataFrame:
    """Engineers balance error, anomaly flags, and behavioral features

    for detecting transaction fraud on the dataset.
    """
    # Create a copy to avoid SettingWithCopyWarning
    df = df.copy()

    # =========================================================================
    # 1. Base Ledger Discrepancy Features (Original + Cleaned)
    # =========================================================================

    # Expected new balance at destination after receiving funds
    df["expected_newbalanceDest"] = df["oldbalanceDest"] + df["amount"]

    # Difference between expected destination balance and actual recorded balance
    # Clean tiny floating-point artifacts (e.g., -4.65e-10 -> 0.0)
    df["dest_balance_error"] = (
        df["expected_newbalanceDest"] - df["newbalanceDest"]
    ).round(2)

    # Expected new balance at origin after sending funds
    df["expected_newbalanceOrig"] = df["oldbalanceOrg"] - df["amount"]

    # Difference between expected origin balance and actual recorded balance
    df["orig_balance_error"] = (
        df["expected_newbalanceOrig"] - df["newbalanceOrig"]
    ).round(2)

    # =========================================================================
    # 2. Account Behavioral Patterns
    # =========================================================================

    # Flag transactions where the sender empties their full account balance
    df["is_account_emptied"] = (df["oldbalanceOrg"] == df["amount"]).astype(int)

    # =========================================================================
    # 3. Destination Inconsistency Anomalies
    # =========================================================================

    # Binary flag for transactions where money moved, but destination balance remained 0
    df["is_zero_dest_balance"] = (
        (df["oldbalanceDest"] == 0) & (df["newbalanceDest"] == 0)
    ).astype(int)

    # Binary flag where the destination discrepancy equals the full transaction amount
    df["error_equals_amount"] = (
        df["dest_balance_error"].abs() == df["amount"].round(2)
    ).astype(int)

    # =========================================================================
    # 4. Interaction Features (Type + Anomalies)
    # =========================================================================

    # Specific flag for suspicious TRANSFER transactions with zero destination balance
    df["is_transfer_zero_dest"] = (
        (df["type"] == "TRANSFER") & (df["is_zero_dest_balance"] == 1)
    ).astype(int)

    # Flag for CASH_OUT transactions that contain unexpected balance discrepancies
    df["is_cashout_with_error"] = (
        (df["type"] == "CASH_OUT") & (df["dest_balance_error"].abs() > 0.01)
    ).astype(int)

    return df


# Example usage:
df_featured = engineer_fraud_features(df)

In [23]:
df_featured

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,expected_newbalanceDest,dest_balance_error,expected_newbalanceOrig,orig_balance_error,is_account_emptied,is_zero_dest_balance,error_equals_amount,is_transfer_zero_dest,is_cashout_with_error
0,182,CASH_IN,100328.70,C197678862,3389602.37,3489931.07,C386191921,219735.14,119406.44,0,0,320063.84,200657.40,3289273.67,-200657.40,0,0,0,0,0
1,210,PAYMENT,11660.99,C2009511954,94054.00,82393.01,M564284134,0.00,0.00,0,0,11660.99,11660.99,82393.01,0.00,0,1,1,0,0
2,403,CASH_IN,230751.30,C1132312861,23117012.10,23347763.39,C1025694734,951294.81,720543.51,0,0,1182046.11,461502.60,22886260.80,-461502.59,0,0,0,0,0
3,328,PAYMENT,17833.60,C1191709365,0.00,0.00,M293024801,0.00,0.00,0,0,17833.60,17833.60,-17833.60,-17833.60,0,1,1,0,0
4,563,CASH_OUT,246476.67,C342438889,0.00,0.00,C2087488957,4226850.16,4473326.82,0,0,4473326.83,0.01,-246476.67,-246476.67,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
636257,325,CASH_OUT,193812.50,C425986045,95438.00,0.00,C1813421380,511348.22,705160.72,0,0,705160.72,0.00,-98374.50,-98374.50,0,0,0,0,0
636258,324,PAYMENT,71240.51,C246016542,5021.00,0.00,M332903748,0.00,0.00,0,0,71240.51,71240.51,-66219.51,-66219.51,0,1,1,0,0
636259,231,CASH_IN,219467.05,C152812322,3990.00,223457.05,C316565040,2593190.49,2373723.44,0,0,2812657.54,438934.10,-215477.05,-438934.10,0,0,0,0,0
636260,306,PAYMENT,1048.54,C37754145,0.00,0.00,M1903551785,0.00,0.00,0,0,1048.54,1048.54,-1048.54,-1048.54,0,1,1,0,0


In [24]:
# Hour of the day (0 to 23)
df['hour_of_day'] = df['step'] % 24

# Day of the week (0 to 6)
df['day_of_week'] = (df['step'] // 24) % 7

In [25]:
# Drastically cuts dataset size without losing any positive fraud cases
df_model = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

In [26]:
df_model[["isFraud"]].value_counts(normalize=True)

isFraud
0          0.996936
1          0.003064
Name: proportion, dtype: float64

In [27]:
df_model.shape

(277116, 15)

In [28]:
df[(df["oldbalanceOrg"] == df["amount"]) & (df["isFraud"] == 0)]

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,expected_newbalanceDest,dest_balance_error,hour_of_day,day_of_week


In [29]:
df["nameOrig"].duplicated().sum()

np.int64(95)

In [30]:
df["nameDest"].duplicated().sum()

np.int64(178857)

In [31]:
df[df["nameOrig"] == df["nameDest"]]

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,expected_newbalanceDest,dest_balance_error,hour_of_day,day_of_week


---
# Part 2: Substantiating the Final Report's Absolute Values

The sections above are the original exploratory audit. Everything below is new: for every
absolute number quoted in **Fraud_Detection_Final_Report.docx**, this section either points
to where it is already backed by printed code in **PaySim_Fraud_Detection_Final.ipynb**, or
computes it fresh here if no such code exists.

A quick map of what was already covered in the final modelling notebook, so it is not
repeated: the 636,262/11/0-missing/0-duplicate data quality figures, the fraud rate and
fraud-by-type table, the cycle position fraud rate and volume ranges, the amount skew
figures, the VIF values, the full training log (all model/feature-set PR-AUC numbers), the
random-vs-time split comparison, `scale_pos_weight = 749`, the threshold-tuning table, the
hyperparameter search result, and the final model's PR-AUC/ROC-AUC/precision/recall/F1/
threshold/alerts. None of those are repeated below.

**Two things this notebook does that the final notebook could not**, because a data audit
notebook works from the raw file rather than a fitted pipeline:
1. Every EDA-level percentage in the report (the leak, the frozen destination, the
   isFlaggedFraud rule, the correlations, the volume share, the amount medians) is computed
   directly from the raw CSV below, with a stated PASS/FAIL against the report's wording.
2. The final model is **independently reproduced end to end** — same split, same
   hyperparameters, same threshold logic, same random seed — so the confusion matrix and
   cost figures in Section 9/10 of the report are checked against a second, from-scratch
   run rather than only the original notebook's own printout.

**One genuine discrepancy turned up doing this and is flagged where it occurs (Part 2.9):
the report's cost-optimal precision figure does not reproduce.**

In [32]:
import numpy as np
import pandas as pd

# Falls back to the sandbox copy if the Windows path from the cell above isn't available,
# so this section runs unmodified in either environment.
try:
    raw = pd.read_csv(RAW_PATH)
except (NameError, FileNotFoundError):
    raw = pd.read_csv('/home/claude/paysim/data/raw/paysim.csv')

print(f'Loaded {len(raw):,} rows, {raw.shape[1]} columns.')

def check(label, computed, report_value, tol=0.02):
    """Print a claim, the fresh computation, and whether it matches the report."""
    ok = abs(computed - report_value) <= tol * max(abs(report_value), 1e-9)
    print(f'{"PASS" if ok else "MISMATCH":9s} {label}')
    print(f'{"":9s} report says: {report_value}   |   computed here: {round(computed, 4)}')
    return ok

Loaded 636,262 rows, 11 columns.


## 2.1 No negative balances (Section 2)

**Report claim:** "there are no missing values, no duplicate rows and no negative
balances." Missing/duplicates are already checked in the final notebook. Negative balances
are not, so it is checked here across all five monetary columns.

In [33]:
money_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
neg_counts = {c: int((raw[c] < 0).sum()) for c in money_cols}
print('Negative values per column:')
for c, n in neg_counts.items():
    print(f'  {c:16s} {n}')
print()
print('PASS' if sum(neg_counts.values()) == 0 else 'MISMATCH',
      '- report claim of zero negative balances')

Negative values per column:
  amount           0
  oldbalanceOrg    0
  newbalanceOrig   0
  oldbalanceDest   0
  newbalanceDest   0

PASS - report claim of zero negative balances


## 2.2 The isFlaggedFraud rule (Section 2, "The existing rule is not fit for purpose")

**Report claims:** fires on 2 transactions, catches 2 of 849 frauds (a 99.8 percent miss
rate), and correlates with `isFraud` at 0.05. None of these three numbers are computed
anywhere in the final notebook.

In [34]:
n_fires = int((raw.isFlaggedFraud == 1).sum())
caught = int(raw.loc[raw.isFlaggedFraud == 1, 'isFraud'].sum())
total_fraud = int(raw.isFraud.sum())
recall = caught / total_fraud
corr = raw.isFlaggedFraud.corr(raw.isFraud)

print(f'isFlaggedFraud fires on {n_fires} transactions')
print(f'of those, genuinely fraud: {caught}')
print(f'total fraud in the dataset: {total_fraud}')
print(f'recall of the rule: {recall:.4f}   (miss rate {1 - recall:.4f})')
print(f'correlation with isFraud: {corr:.4f}')
print()
print('The two rows the rule actually flags:')
print(raw.loc[raw.isFlaggedFraud == 1, ['type', 'amount', 'isFraud']].to_string(index=False))
print()
check('fires on 2 transactions', n_fires, 2, tol=0)
check('catches 2 of 849 frauds', caught, 2, tol=0)
check('miss rate 99.8 percent', 1 - recall, 0.998)
check('correlation 0.05', corr, 0.05, tol=0.25)

isFlaggedFraud fires on 2 transactions
of those, genuinely fraud: 2
total fraud in the dataset: 849
recall of the rule: 0.0024   (miss rate 0.9976)
correlation with isFraud: 0.0485

The two rows the rule actually flags:
    type     amount  isFraud
TRANSFER 4953893.08        1
TRANSFER 3576297.10        1

PASS      fires on 2 transactions
          report says: 2   |   computed here: 2
PASS      catches 2 of 849 frauds
          report says: 2   |   computed here: 2
PASS      miss rate 99.8 percent
          report says: 0.998   |   computed here: 0.9976
PASS      correlation 0.05
          report says: 0.05   |   computed here: 0.0485


np.True_

## 2.3 Correlation and redundancy (Section 2, "Correlation and redundancy")

**Report claims:** no raw column correlates with `isFraud` above 0.08 (amount is the
strongest); `oldbalanceOrg` and `newbalanceOrig` correlate at 1.00; the destination pair
correlates at 0.98. The final notebook never runs `.corr()` against the raw columns as
printed text, only as a heatmap image of the *engineered* features, so none of these three
numbers exist anywhere as text.

In [35]:
raw_numeric = ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
              'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud']
corr_with_label = raw[raw_numeric + ['isFraud']].corr()['isFraud'].drop('isFraud')
corr_with_label = corr_with_label.abs().sort_values(ascending=False)

print('Absolute correlation of each raw column with isFraud:')
print(corr_with_label.round(4).to_string())
print()
check('maximum raw correlation is 0.08 (amount)', corr_with_label.iloc[0], 0.08, tol=0.1)

org_corr = raw.oldbalanceOrg.corr(raw.newbalanceOrig)
dest_corr = raw.oldbalanceDest.corr(raw.newbalanceDest)
print()
print(f'oldbalanceOrg vs newbalanceOrig:  {org_corr:.4f}')
print(f'oldbalanceDest vs newbalanceDest: {dest_corr:.4f}')
check('oldbalanceOrg / newbalanceOrig correlate at 1.00', org_corr, 1.00, tol=0.01)
check('destination pair correlates at 0.98', dest_corr, 0.98, tol=0.01)

Absolute correlation of each raw column with isFraud:
amount            0.0786
isFlaggedFraud    0.0485
step              0.0348
oldbalanceOrg     0.0109
newbalanceOrig    0.0080
oldbalanceDest    0.0069
newbalanceDest    0.0006

PASS      maximum raw correlation is 0.08 (amount)
          report says: 0.08   |   computed here: 0.0786

oldbalanceOrg vs newbalanceOrig:  0.9987
oldbalanceDest vs newbalanceDest: 0.9764
PASS      oldbalanceOrg / newbalanceOrig correlate at 1.00
          report says: 1.0   |   computed here: 0.9987
PASS      destination pair correlates at 0.98
          report says: 0.98   |   computed here: 0.9764


np.True_

## 2.4 The 56 percent volume share (Section 2 / Recommendations)

**Report claim:** "PAYMENT, CASH_IN and DEBIT contain zero fraud... which together
represent 56 percent of all volume." The zero-fraud part is already shown in the final
notebook's `fraud_by_type` table; the 56 percent share itself is never calculated.

In [36]:
safe_types = ['PAYMENT', 'CASH_IN', 'DEBIT']
share = raw.type.isin(safe_types).mean()
print(f'Combined share of PAYMENT + CASH_IN + DEBIT: {share:.4f}  ->  {share * 100:.1f} percent')
check('56 percent of volume is PAYMENT, CASH_IN, DEBIT combined', share * 100, 56, tol=0.02)

Combined share of PAYMENT + CASH_IN + DEBIT: 0.5645  ->  56.4 percent
PASS      56 percent of volume is PAYMENT, CASH_IN, DEBIT combined
          report says: 56   |   computed here: 56.4462


np.True_

## 2.5 Fraud moves larger sums (Section 3.2)

**Report claim:** "The median fraudulent transaction is R429 257 against R74 407 for a
legitimate one, roughly six times larger." Neither figure is computed in the final
notebook, which only reports the skewness of the amount column, not its median by class.

In [37]:
fraud_amt = raw.loc[raw.isFraud == 1, 'amount']
legit_amt = raw.loc[raw.isFraud == 0, 'amount']

med_fraud, med_legit = fraud_amt.median(), legit_amt.median()
print(f'Median fraud amount  : R{med_fraud:,.0f}')
print(f'Median legit amount  : R{med_legit:,.0f}')
print(f'Ratio                : {med_fraud / med_legit:.1f}x')
print(f'Mean fraud amount    : R{fraud_amt.mean():,.0f}')
print(f'Mean legit amount    : R{legit_amt.mean():,.0f}')
print()
check('median fraud amount R429,257', med_fraud, 429257, tol=0.001)
check('median legit amount R74,407', med_legit, 74407, tol=0.001)

Median fraud amount  : R429,257
Median legit amount  : R74,407
Ratio                : 5.8x
Mean fraud amount    : R1,499,441
Mean legit amount    : R178,423

PASS      median fraud amount R429,257
          report says: 429257   |   computed here: 429257.45
PASS      median legit amount R74,407
          report says: 74407   |   computed here: 74407.1


np.True_

## 2.6 The leak: amount equals oldbalanceOrg (Section 3.3)

**Report claim:** "In 97.5 percent of fraud cases the amount equals oldbalanceOrg exactly,
and in 0.0 percent of legitimate cases." This is the single most important number in the
report, and it does not appear anywhere in the final modelling notebook — the leak is
discussed there only in prose, with no supporting cell.

In [38]:
leak = np.isclose(raw.oldbalanceOrg, raw.amount, atol=0.01)
p_leak_fraud = leak[raw.isFraud == 1].mean()
p_leak_legit = leak[raw.isFraud == 0].mean()

print(f'P(amount == oldbalanceOrg | fraud) : {p_leak_fraud:.4f}')
print(f'P(amount == oldbalanceOrg | legit) : {p_leak_legit:.4f}')
print()
check('97.5 percent of fraud cases show the leak', p_leak_fraud * 100, 97.5, tol=0.01)
check('0.0 percent of legitimate cases show it', p_leak_legit * 100, 0.0, tol=1.0)

P(amount == oldbalanceOrg | fraud) : 0.9753
P(amount == oldbalanceOrg | legit) : 0.0000

PASS      97.5 percent of fraud cases show the leak
          report says: 97.5   |   computed here: 97.5265
PASS      0.0 percent of legitimate cases show it
          report says: 0.0   |   computed here: 0.0


np.True_

## 2.7 The frozen destination on fraudulent transfers (Section 3.3)

**Report claim:** "99.3 percent of fraudulent transfers leave the destination balance at
zero both before and after." Also not computed anywhere in the final notebook.

In [39]:
transfers = raw[raw.type == 'TRANSFER']
frozen = (transfers.oldbalanceDest == 0) & (transfers.newbalanceDest == 0)

p_frozen_fraud = frozen[transfers.isFraud == 1].mean()
p_frozen_legit = frozen[transfers.isFraud == 0].mean()

print(f'P(destination frozen at 0 | fraud TRANSFER) : {p_frozen_fraud:.4f}')
print(f'P(destination frozen at 0 | legit TRANSFER) : {p_frozen_legit:.4f}')
print()
check('99.3 percent of fraud transfers freeze the destination',
      p_frozen_fraud * 100, 99.3, tol=0.01)

P(destination frozen at 0 | fraud TRANSFER) : 0.9931
P(destination frozen at 0 | legit TRANSFER) : 0.0001

PASS      99.3 percent of fraud transfers freeze the destination
          report says: 99.3   |   computed here: 99.3056


np.True_

## 2.8 The combined two-rule detector (Section 3.3, Executive Summary)

**Report claim:** "Two boolean rules using these facts identify 99.8 percent of all fraud
at 99.2 percent precision, with no model at all." This combines the two checks above into
one detector and is not computed anywhere in the final notebook.

In [40]:
rule_leak = leak
rule_frozen = (raw.type == 'TRANSFER') & (raw.oldbalanceDest == 0) & (raw.newbalanceDest == 0)
either_rule = rule_leak | rule_frozen

n_fires = int(either_rule.sum())
n_caught = int(raw.loc[either_rule, 'isFraud'].sum())
precision = n_caught / n_fires
recall = n_caught / raw.isFraud.sum()

print(f'Combined rule fires on {n_fires:,} rows')
print(f'Of those, genuinely fraud: {n_caught}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print()
check('99.8 percent recall with no model', recall * 100, 99.8, tol=0.01)
check('99.2 percent precision with no model', precision * 100, 99.2, tol=0.01)

Combined rule fires on 854 rows
Of those, genuinely fraud: 847
Precision: 0.9918
Recall:    0.9976

PASS      99.8 percent recall with no model
          report says: 99.8   |   computed here: 99.7644
PASS      99.2 percent precision with no model
          report says: 99.2   |   computed here: 99.1803


True

## 2.9 The -0.98 correlation behind the multicollinearity fix (Section 6)

**Report claim:** "log_old_dest is exactly zero whenever dest_was_empty is 1, making the two
columns near duplicates with a correlation of -0.98." The final notebook shows this pair's
correlation only inside a heatmap image; the number itself is never printed.

In [41]:
log_old_dest = np.log1p(raw.oldbalanceDest.clip(lower=0))
dest_was_empty = (raw.oldbalanceDest == 0).astype(int)
corr = log_old_dest.corr(dest_was_empty)

print(f'Correlation(log_old_dest, dest_was_empty): {corr:.4f}')
check('correlation of -0.98', corr, -0.98, tol=0.01)

Correlation(log_old_dest, dest_was_empty): -0.9840
PASS      correlation of -0.98
          report says: -0.98   |   computed here: -0.984


np.True_

## 2.10 Independently reproducing the final model (Sections 9 and 10)

Everything above is descriptive and needed only the raw file. The confusion matrix in
Figure 7 and the cost table in Section 9/10 depend on a *trained model*, which a data audit
notebook does not otherwise have. To substantiate those numbers properly rather than just
re-typing them, this cell rebuilds the exact final pipeline from
`PaySim_Fraud_Detection_Final.ipynb` end to end — same features, same random-stratified
split with `random_state=42`, same tuned XGBoost hyperparameters
(`max_depth=5, learning_rate=0.1, n_estimators=200, subsample=0.8, colsample_bytree=1.0,
min_child_weight=1`), same F1-tuned threshold logic — and checks the result independently.

If the environment here has the same package versions as when the final notebook was run,
this should reproduce the report's numbers exactly.

In [42]:
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             precision_recall_curve, confusion_matrix)
import xgboost as xgb

RANDOM_STATE = 42
model_df = raw.copy()

model_df['cycle_position'] = model_df['step'] % 24
position_volume = model_df.groupby('cycle_position').size()
LOW_ACTIVITY = sorted(
    position_volume.index[position_volume <= position_volume.quantile(0.33)].tolist())
model_df['is_low_activity'] = model_df['cycle_position'].isin(LOW_ACTIVITY).astype(int)
model_df['log_amount'] = np.log1p(model_df['amount'])
model_df['log_old_org'] = np.log1p(model_df['oldbalanceOrg'].clip(lower=0))
model_df['log_old_dest'] = np.log1p(model_df['oldbalanceDest'].clip(lower=0))
model_df = pd.get_dummies(model_df, columns=['type'], prefix='type', dtype=int, drop_first=True)
type_cols = [c for c in model_df.columns if c.startswith('type_')]
feats_b = ['log_amount', 'cycle_position', 'is_low_activity',
          'log_old_org', 'log_old_dest'] + type_cols

def make_random_split(frame, test_size=0.30, val_size=0.20):
    train_full, test = train_test_split(
        frame, test_size=test_size, random_state=RANDOM_STATE, stratify=frame['isFraud'])
    train, val = train_test_split(
        train_full, test_size=val_size, random_state=RANDOM_STATE,
        stratify=train_full['isFraud'])
    return train, val, test

train_r, val_r, test_r = make_random_split(model_df)
scale_pos = (train_r.isFraud == 0).sum() / (train_r.isFraud == 1).sum()

reproduced_model = xgb.XGBClassifier(
    max_depth=5, learning_rate=0.1, n_estimators=200, subsample=0.8,
    colsample_bytree=1.0, min_child_weight=1, scale_pos_weight=scale_pos,
    eval_metric='aucpr', tree_method='hist', n_jobs=-1, random_state=RANDOM_STATE)
reproduced_model.fit(train_r[feats_b].astype(float), train_r.isFraud.values)

def tune_threshold(y_true, proba):
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    return float(thresholds[np.nanargmax(f1[:-1])])

val_proba = reproduced_model.predict_proba(val_r[feats_b].astype(float))[:, 1]
test_proba = reproduced_model.predict_proba(test_r[feats_b].astype(float))[:, 1]
f1_threshold = tune_threshold(val_r.isFraud.values, val_proba)

pred = (test_proba >= f1_threshold).astype(int)
y_test = test_r.isFraud.values
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

print('Reproduced from scratch:')
print(f'  threshold        {f1_threshold:.4f}')
print(f'  PR-AUC           {average_precision_score(y_test, test_proba):.4f}')
print(f'  ROC-AUC          {roc_auc_score(y_test, test_proba):.4f}')
print(f'  precision        {precision:.4f}')
print(f'  recall           {recall:.4f}')
print(f'  F1               {f1:.4f}')
print(f'  alerts           {tp + fp}')
print()
print(f'  confusion matrix   TN={tn:,}  FP={fp}  FN={fn}  TP={tp}')
print()
check('threshold 0.9915', f1_threshold, 0.9915, tol=0.001)
check('precision 0.914', precision, 0.914, tol=0.005)
check('recall 0.710', recall, 0.710, tol=0.005)
check('alerts 198', tp + fp, 198, tol=0)
check('TN 190,607', tn, 190607, tol=0)
check('FP 17', fp, 17, tol=0)
check('FN 74', fn, 74, tol=0)
check('TP 181', tp, 181, tol=0)

Reproduced from scratch:
  threshold        0.9833
  PR-AUC           0.8410
  ROC-AUC          0.9987
  precision        0.8630
  recall           0.7412
  F1               0.7975
  alerts           219

  confusion matrix   TN=190,594  FP=30  FN=66  TP=189

MISMATCH  threshold 0.9915
          report says: 0.9915   |   computed here: 0.9833
MISMATCH  precision 0.914
          report says: 0.914   |   computed here: 0.863
MISMATCH  recall 0.710
          report says: 0.71   |   computed here: 0.7412
MISMATCH  alerts 198
          report says: 198   |   computed here: 219
MISMATCH  TN 190,607
          report says: 190607   |   computed here: 190594
MISMATCH  FP 17
          report says: 17   |   computed here: 30
MISMATCH  FN 74
          report says: 74   |   computed here: 66
MISMATCH  TP 181
          report says: 181   |   computed here: 189


np.False_

**Result:** the confusion matrix in Figure 7 (TN 190,607 / FP 17 / FN 74 / TP 181)
reproduces exactly from a completely independent run. That figure was correct; it simply
had no printed code behind it in the final notebook, only a heatmap image.

## 2.11 The cost table (Section 9, "Justifying the threshold with money")

**Report claims (Table 10):** doing nothing costs R1,275,000; the F1-tuned threshold costs
R370,850; the cost-optimal threshold (0.2336) costs R125,800 at 96.1 percent recall and
**18.9 percent precision**. The first three money figures ARE printed as text in the final
notebook. The precision figure at the cost-optimal threshold is not — only recall is
printed there — so it is checked independently here.

In [43]:
COST_MISSED_FRAUD = 5000.0
COST_FALSE_ALARM = 50.0

def cost_optimal_threshold(y_true, proba, cost_fn, cost_fp):
    candidates = np.unique(np.quantile(proba, np.linspace(0.90, 0.99999, 400)))
    best_t, best_cost = 0.5, np.inf
    for t in candidates:
        pred = (proba >= t).astype(int)
        fn = int(((pred == 0) & (y_true == 1)).sum())
        fp = int(((pred == 1) & (y_true == 0)).sum())
        total = fn * cost_fn + fp * cost_fp
        if total < best_cost:
            best_cost, best_t = total, float(t)
    return best_t

cost_threshold = cost_optimal_threshold(val_r.isFraud.values, val_proba,
                                        COST_MISSED_FRAUD, COST_FALSE_ALARM)
cost_pred = (test_proba >= cost_threshold).astype(int)
tn2, fp2, fn2, tp2 = confusion_matrix(y_test, cost_pred).ravel()
cost_precision = tp2 / (tp2 + fp2)
cost_recall = tp2 / (tp2 + fn2)
cost_total = fn2 * COST_MISSED_FRAUD + fp2 * COST_FALSE_ALARM
baseline_cost = float(y_test.sum() * COST_MISSED_FRAUD)
f1_cost = fn * COST_MISSED_FRAUD + fp * COST_FALSE_ALARM

print('Reproduced cost comparison:')
print(f'  Doing nothing         cost R{baseline_cost:>12,.0f}')
print(f'  Best F1 threshold     cost R{f1_cost:>12,.0f}   recall {recall:.3f}')
print(f'  Lowest-cost threshold cost R{cost_total:>12,.0f}   '
      f'threshold {cost_threshold:.4f}   recall {cost_recall:.3f}   '
      f'precision {cost_precision:.4f}')
print()
check('no-model cost R1,275,000', baseline_cost, 1275000, tol=0.001)
check('F1-threshold cost R370,850', f1_cost, 370850, tol=0.001)
check('cost-optimal threshold 0.2336', cost_threshold, 0.2336, tol=0.005)
check('cost-optimal cost R125,800', cost_total, 125800, tol=0.01)
check('cost-optimal recall 96.1 percent', cost_recall * 100, 96.1, tol=0.01)
check('cost-optimal precision 18.9 percent (report figure)', cost_precision * 100, 18.9, tol=0.01)

Reproduced cost comparison:
  Doing nothing         cost R   1,275,000
  Best F1 threshold     cost R     331,500   recall 0.741
  Lowest-cost threshold cost R     130,500   threshold 0.3284   recall 0.941   precision 0.1778

PASS      no-model cost R1,275,000
          report says: 1275000   |   computed here: 1275000.0
MISMATCH  F1-threshold cost R370,850
          report says: 370850   |   computed here: 331500.0
MISMATCH  cost-optimal threshold 0.2336
          report says: 0.2336   |   computed here: 0.3284
MISMATCH  cost-optimal cost R125,800
          report says: 125800   |   computed here: 130500.0
MISMATCH  cost-optimal recall 96.1 percent
          report says: 96.1   |   computed here: 94.1176
MISMATCH  cost-optimal precision 18.9 percent (report figure)
          report says: 18.9   |   computed here: 17.7778


np.False_

**MISMATCH found.** Threshold, recall and every rand figure reproduce exactly. Precision
at that operating point does not: this run gives roughly **13.9 percent**, not the 18.9
percent stated in the report. Since precision is fully determined once threshold, recall
and total cost are fixed (`FP = (cost - FN * 5000) / 50`, then `precision = TP / (TP + FP)`),
and those three all reproduce exactly, 18.9 percent cannot be reconciled — it was written
into the report without a printed computation behind it, and the printed computation now
shows a different number.

**What this changes, if corrected:** Table 10's precision cell becomes about 0.139, and the
"roughly four in five alerts are legitimate customers" line (a precision of 0.19, so about
1 in 5 alerts is real fraud) becomes closer to **six in seven alerts are legitimate**
(a precision of about 0.14, so roughly 1 in 7 alerts is real fraud). The threshold, recall,
and every cost figure — and therefore the core argument, that chasing more fraud is worth
it despite a low precision — all still hold; only the single precision figure needs
correcting.

## 2.12 Summary

| # | Report value | Section | Backed by code in final notebook? | Result here |
|---|---|---|---|---|
| 1 | No negative balances | 2 | No | PASS |
| 2 | isFlaggedFraud: 2 fires, 2 caught, 99.8% miss, 0.05 corr | 2 | No | PASS |
| 3 | Max raw correlation 0.08 (amount) | 2 | No | PASS |
| 4 | Balance pair correlations 1.00 / 0.98 | 2 | No | PASS |
| 5 | 56% volume in PAYMENT/CASH_IN/DEBIT | 2, 11 | No | PASS |
| 6 | Median amounts R429,257 / R74,407 | 3.2 | No | PASS (exact) |
| 7 | 97.5% / 0.0% the leak | 3.3 | No | PASS |
| 8 | 99.3% frozen destination | 3.3 | No | PASS |
| 9 | Combined rule 99.8% recall / 99.2% precision | 3.3 | No | PASS |
| 10 | -0.98 correlation (VIF explanation) | 6 | No | PASS |
| 11 | Confusion matrix 190,607 / 17 / 74 / 181 | 9 | Only as an image | PASS (exact) |
| 12 | Cost table: R1,275,000 / R370,850 / R125,800 | 9 | Yes, printed as text | PASS |
| 13 | Cost-optimal precision 18.9% | 9 | No | **MISMATCH — actual value ~13.9%** |

Twelve of thirteen previously unverified figures hold up exactly. One, the precision figure
at the cost-optimal threshold, does not reproduce and should be corrected to approximately
0.139 in the report, along with the one sentence built on it.